# 03 — Full pipeline analysis

**Inputs:** one external development PLY, compatible Run 1 `.pt`, manifest sample ID where applicable, and a **new** output directory. Install `.[geometry,analysis,visualization]` for optional HTML. Set the inputs below and run top-to-bottom.

This uses the same frozen composition as `python -m agri3d.pipeline analyze`: semantic inference → full-cloud projection → geometric components → metric phenotype proxies → artifact writer. Instance settings are 0.020 m / 32 neighbors, minimum 100 points, attachment radius 0.03 m. No tuning, training or test evaluation occurs.

Component counts are not verified biological leaf counts. Length and volume measurements remain geometric proxies. Local artifacts can include input paths; sanitize them and review dataset terms before sharing. CI supplies only synthetic fixtures and a synthetic backend.


In [ ]:
from pathlib import Path

# Set these public inputs before running all cells. Never supply test plants.
# The optional context is used only by the documented synthetic CI smoke test.
CONTEXT = globals().get("_AGRI3D_NOTEBOOK_CONTEXT", {})
PLY_PATH = CONTEXT.get("ply_path", None)
SHOW_VIEWS = CONTEXT.get("visualize", True)
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise ValueError("Launch this notebook from the checkout or its notebooks directory")
if PLY_PATH is None:
    raise ValueError("Set PLY_PATH to an external development PLY, then run all cells")
PLY_PATH = Path(PLY_PATH)

CHECKPOINT = CONTEXT.get("checkpoint", None)
SAMPLE_ID = CONTEXT.get("sample_id", "example-plant")  # use the manifest ID for reproduction
CONFIG = PROJECT_ROOT / "configs" / "phenotyping_pipeline.yaml"
DEVICE = "cpu"

OUTPUT_DIR = CONTEXT.get("output_dir", None)
if OUTPUT_DIR is None:
    raise ValueError("Set OUTPUT_DIR to a new external artifact directory")
OUTPUT_DIR = Path(OUTPUT_DIR)


In [ ]:
from agri3d.inference.pointnet2_backend import PyTorchPointNet2Predictor
from agri3d.pipeline.analyze import analyze_plant
from agri3d.pipeline.config import PipelineConfig

predictor = CONTEXT.get("predictor")  # synthetic smoke-test backend only; normally unset
if predictor is None:
    if CHECKPOINT is None:
        raise ValueError("Set CHECKPOINT to a compatible external Run 1 best.pt")
    predictor = PyTorchPointNet2Predictor(CHECKPOINT, device=DEVICE)
result = analyze_plant(PLY_PATH, predictor=predictor,
                       config=PipelineConfig.load(CONFIG), sample_id=SAMPLE_ID)


In [ ]:
from agri3d.pipeline.artifacts import write_plant

write_plant(OUTPUT_DIR, result, visualize=SHOW_VIEWS)
print("Component count (not biological leaf count):", len(result.structure.leaves))
print("Phenotype proxies:", result.report["phenotypes"])
print("Saved artifacts:", sorted(p.name for p in OUTPUT_DIR.iterdir()))
assert (OUTPUT_DIR / "phenotype.json").is_file()
assert (OUTPUT_DIR / "structure.ply").is_file()
